## Model core

1. Portfolio expected return
$$E(R_{p}) = \sum_{i=1}^nw_{i}E(R_{i}),$$
where $w_{i}$ - weight, that equals to the share of the securities in the portfolio.

$E(R_{i})$ - return of the particular stock in the portfolio.

2. Portfolio variance

Portfolio dispersion is a process that determines the degree of risk or volatility associated with an investment portfolio. The basic formula for calculating this variance focuses on the relationship between the so-called return variance and the covariation associated with each of the stocks found in the portfolio, as well as the percentage or part of the portfolio that each stock represents.
$$\sigma_{p}^{2} = \sum_{i}^{}\omega_{i}^{2}\sigma_{i}^{2}+\sum_{i}^{}\sum_{j\neq i}^{}\omega_{i}^{}\omega_{j}^{}\sigma_{i}^{}\sigma_{j}^{}\rho_{ij},$$
where $\omega_{i}$ - stock weights; $\sigma_{i}$ - stock return std; $\rho_{ij}$ - correlation coeff. between two stocks.

3. Sharpe ratio
$$\frac{R_{p} - R_{f}}{\sigma_{p}}$$

4. Efficient frontier

<center>
<img src="https://upload.wikimedia.org/wikipedia/commons/e/e1/Markowitz_frontier.jpg?utm_source=en.wikipedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled" width="400" height="200" alt="Efficient frontier">
</center>

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
while project_root.name != 'python' and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Downloading and cleaning up the data

In [ ]:
from data import download_tickers_history

TRADING_DAYS_PER_YEAR = 252

# set the date range for the historic data
start_date = datetime(year=2022, month=1, day=1)
end_date = datetime(year=2026, month=1, day=1)
tickers = ['NVDA', 'AAPL', 'PLTR', 'DKNG', 'CAT', 'INTC', 'AMZN', 'TSLA', 'GOOG', 'MSFT']
history = download_tickers_history(start_date, end_date, tickers)

print(history.isnull().sum())


## Optimization
$$\min_{w} \quad \frac{1}{2} w^T \Sigma w \quad \text{(minimizing portfolio variance)},$$
where $\Sigma$ is a covariate matrix of asset returns (in the size $N$ times $N$),

and with the following constraints:
1. $w^T \mathbf{1} = 1$ - the sum of all weights equals 100% (all capital is distributed).
2. $\mu = \mu_{\text{target}}$ - the portfolio must deliver exactly the return we have fixed.
3. If shorts are prohibited, the limit-inequality is added: $w_i \ge 0 \quad \forall i$ (Long-only).
---

Risk-Free Rate ($R_f$) - is a risk-free interest rate (the yield of an asset with zero credit and market risk).

The difference $(R_p - R_f)$ in the numeral is called excess return or risk premium.
The risk-free rate shows, "How much can I earn without risking my money at all?"

What to take as $R_f$?
* For USD:
    - U.S. 13-Week Treasury Bills (T-Bills) (_^IRX_ ticker) - for daily trading or if the portfolio is rebalanced more than twice a year
    - U.S. 10-Year Treasury Notes (_^TNX_ ticker) - for long-term trading
    - SOFR (Secured Overnight Financing Rate)
* For EUR:
    - Germany Government Bonds with AAA rating

In [ ]:
from src.portfolio.markowitz import get_risk_free_rate

start_date = datetime(year=2022, month=1, day=1)
end_date = datetime(year=2026, month=1, day=1)
risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)

print(f'{risk_free_rate:.3%}')


We will apply numeric convex optimization method; more specifically - Sequential Least Squares Programming (**SLSQP**) - standard gradient descent with constraints (implemented in _scipy.optimize.miminize_)

In [ ]:
from src.portfolio import gmf_return_optimization

optimum_df = gmf_return_optimization(tickers_df=history, rf_base='T_BILLS')
optimum_df


Now we will optimize Sharpe ratio directly, since this is the only point where the Efficient Frontier touches the Capital Market Line (CML), thus has the most optimal stocks weights distribution (portfolio). This point is called Tangency Portfolio.

In [ ]:
from src.portfolio import find_max_sharpe

(max_sharpe, stocks_w) = find_max_sharpe(tickers_df=history, rf_base='T_BILLS')

exact_max_ret = max_sharpe.tangency_return
exact_max_vol = max_sharpe.tangency_vol
exact_max_sharpe = max_sharpe.max_sharpe

print(f"Exact Max Sharpe Ratio: {max_sharpe.max_sharpe:.4f}")
print(f"Exact Tangency Return: {exact_max_ret:.2%}")
print(f"Exact Tangency Volatility: {exact_max_vol:.2%}")

print("Exact optimum stocks distribution:")
stocks_w


### Visualize Efficient Frontier

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

vol = optimum_df['vol']
sharpe = optimum_df['sharpe']
returns = optimum_df['return']
max_vol = vol.max()
max_sharpe = sharpe.max()
max_sharpe_ind = sharpe.idxmax()

# capital distribution line
cal_x = np.linspace(0, max_vol, 100)
cal_y = max_sharpe * cal_x + risk_free_rate

ax.plot(
    cal_x,
    cal_y,
    color='darkblue',
    alpha=0.7,
    linewidth=2, 
)
ax.spines['left'].set_position('zero')
x_label = max_vol * 0.85
y_label = max_sharpe * x_label + risk_free_rate

# Add text on the plot
ax.text(
    x_label,
    y_label + 0.02,
    'Capital Market Line',
    color='darkblue',
    fontweight='bold'
)

# mark risk free rate point
plt.plot(0, risk_free_rate, marker="o", markersize=8, markeredgecolor="red", markerfacecolor="yellow")
plt.annotate(
    'Risk free rate',
    xy=(0, risk_free_rate),
    xytext=(risk_free_rate + 0.01, risk_free_rate + 0.2),
    arrowprops=dict(facecolor='black', shrink=0.05)
)

# effective frontier
ef_x = vol
ef_y = returns
ax.plot(
    ef_x,
    ef_y,
    color='darkgreen',
    alpha=0.7,
    linewidth=2,
)

x_label = max_vol * 0.85
y_label = returns.max() * 0.85
# Add text on the plot
ax.text(
    x_label,
    y_label + 0.02,
    'Efficient frontier',
    color='darkgreen',
    fontweight='bold'
)

# mark the tangency portfolio point
plt.plot(exact_max_vol, exact_max_ret, marker="*", markersize=8, markerfacecolor="red")
plt.annotate(
    'Tangency portfolio',
    xy=(exact_max_vol, exact_max_ret),
    xytext=(exact_max_vol + 0.01, exact_max_ret - 0.05),
    arrowprops=dict(facecolor='black', shrink=0.02)
)

plt.title('Efficient Frontier')
plt.xlabel('Portfolio deviation', fontsize=12)
plt.ylabel('Expected return', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()


Now instead of hard fixed returns, we will minimize a Utility Function:$$\min_{w} \quad \left[ \frac{1}{2} w^T \Sigma w - \gamma \cdot (w^T \mu) \right]$$where $\gamma$ -parameter of risk acceptance:
* For $\gamma = 0$ the optimizer only cares about risk (finds a portfolio with minimal dispersion).
* For $\gamma \to \infty$ the optimizer only cares about returns (puts all the money into the most profitable asset).

In [ ]:
from data import log_returns

N = history.columns.levels[0].nunique()
log_ret = log_returns(history)
yr_cov = log_ret.cov() * TRADING_DAYS_PER_YEAR  # type: ignore
weights = np.ones(N) / N;
expected_returns = log_ret.mean() * TRADING_DAYS_PER_YEAR


In [ ]:
from scipy.optimize import minimize

def get_target_fun(weights, gamma):
    return 0.5 * (weights.T @ yr_cov @ weights) - gamma * (weights.T @ expected_returns)

constraints = [
    {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
]

# Bounds: Long-only (0 <= w_i <= 1)
bounds = tuple((0, 0.4) for _ in range(N))
gammas = np.logspace(-2, 1, 100)

optimum_results = []
last_optimal_weights = weights
for gamma in gammas:
    res = minimize(
            fun=lambda x: get_target_fun(x, gamma),
            x0=last_optimal_weights,
            method='SLSQP',
            constraints=constraints,
            bounds=bounds
        )

    if res.success: 
        w = res.x
        last_optimal_weights = w
        ret = np.dot(w, expected_returns)
        vol = np.sqrt(w.T @ yr_cov @ w)
        sharpe = (ret - risk_free_rate) / vol

        optimum_results.append({
            'gamma': gamma,
            'weights': w,
            'return': ret,
            'vol': vol,
            'sharpe': sharpe,
        })

optimum_df = pd.DataFrame(optimum_results)
optimum_df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

vol = optimum_df['vol']
sharpe = optimum_df['sharpe']
returns = optimum_df['return']
max_vol = vol.max()
max_sharpe = sharpe.max()
max_sharpe_ind = sharpe.idxmax()

# capital distribution line
cal_x = np.linspace(0, max_vol, 100)
cal_y = max_sharpe * cal_x + risk_free_rate

ax.plot(
    cal_x,
    cal_y,
    color='darkblue',
    alpha=0.7,
    linewidth=2, 
)
ax.spines['left'].set_position('zero')
x_label = max_vol * 0.85
y_label = max_sharpe * x_label + risk_free_rate

# Add text on the plot
ax.text(
    x_label,
    y_label + 0.02,
    'Capital Market Line',
    color='darkblue',
    fontweight='bold'
)

# mark risk free rate point
plt.plot(0, risk_free_rate, marker="o", markersize=8, markeredgecolor="red", markerfacecolor="yellow")
plt.annotate(
    'Risk free rate',
    xy=(0, risk_free_rate),
    xytext=(risk_free_rate + 0.01, risk_free_rate + 0.2),
    arrowprops=dict(facecolor='black', shrink=0.05)
)

# effective frontier
ef_x = vol
ef_y = returns
ax.plot(
    ef_x,
    ef_y,
    color='darkgreen',
    alpha=0.7,
    linewidth=2,
)

# equal weights portfolio
x0 = np.ones(N) / N
def_x = np.sqrt(x0.T @ yr_cov @ x0)
def_y = np.dot(x0, expected_returns)
ax.scatter(
    def_x,
    def_y,
    color='darkred',
    alpha=0.7,
)
plt.text(def_x + 0.001, def_y + 0.01, 'Equal weights portfolio')

# S&P 500 portfolio
sp500_df = download_tickers_history(start_date, end_date, ['^GSPC'])
sp500_log_ret = log_returns(sp500_df)

sp500_x = sp500_log_ret.std().iloc[0] * np.sqrt(TRADING_DAYS_PER_YEAR)
sp500_y = sp500_log_ret.mean().iloc[0] * TRADING_DAYS_PER_YEAR
ax.scatter(
    sp500_x,
    sp500_y,
    color='red',
    alpha=0.7,
)
plt.text(sp500_x + 0.001, sp500_y + 0.01, 'S&P 500 portfolio')

x_label = max_vol * 0.85
y_label = returns.max() * 0.85
# Add text on the plot
ax.text(
    x_label,
    y_label + 0.02,
    'Efficient frontier',
    color='darkgreen',
    fontweight='bold'
)

# mark the tangency portfolio point
plt.plot(exact_max_vol, exact_max_ret, marker="*", markersize=8, markerfacecolor="red")
plt.annotate(
    'Tangency portfolio',
    xy=(exact_max_vol, exact_max_ret),
    xytext=(exact_max_vol + 0.01, exact_max_ret - 0.05),
    arrowprops=dict(facecolor='black', shrink=0.02)
)

plt.title('Efficient Frontier (from Utility Function)')
plt.xlabel('Portfolio deviation', fontsize=12)
plt.ylabel('Expected return', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()
